# Running the experiments on Kaggle

Settings: accelerator GPU T4 x2, internet on, secret `GITHUB_TOKEN` (read access to the repository).

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/omarht27/textgrain-eval.git
%cd /kaggle/working/textgrain-eval
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from textgrain.keys import new_key, save_key

os.makedirs("results", exist_ok=True)
if not os.path.exists("results/secret.key"):
    save_key(new_key(), "results/secret.key")

## Smoke test

In [ ]:
!python experiments/generate.py --model Qwen/Qwen2.5-1.5B-Instruct --method textgrain --n-prompts 4 --batch-size 4 --max-new-tokens 50 --no-eos --out results/smoke.jsonl

## Fixed-length series (one queue of jobs per GPU)

In [ ]:
import subprocess

COMMON = "--model Qwen/Qwen2.5-1.5B-Instruct --n-prompts 200 --batch-size 16 --max-new-tokens 400 --no-eos"
JOBS = {
    0: [
        "--method none --out results/none.jsonl",
        "--method gumbel --out results/gumbel.jsonl",
        "--method greenlist --delta 2.0 --out results/greenlist_d2.jsonl",
        "--method textgrain --beta 0.1 --out results/textgrain_B4_b0.1.jsonl",
        "--method textgrain --beta 0.25 --out results/textgrain_B4_b0.25.jsonl",
    ],
    1: [
        "--method textgrain --beta 0.5 --out results/textgrain_B4_b0.5.jsonl",
        "--method textgrain --beta 0.25 --blocks 2 --columns 2 --out results/textgrain_B2_b0.25.jsonl",
        "--method textgrain --beta 0.25 --blocks 8 --columns 8 --out results/textgrain_B8_b0.25.jsonl",
        "--method textgrain --beta 0.5 --blocks 8 --columns 8 --out results/textgrain_B8_b0.5.jsonl",
        "--method textgrain --beta 0.75 --blocks 8 --columns 8 --out results/textgrain_B8_b0.75.jsonl",
    ],
}

processes = []
for gpu, jobs in JOBS.items():
    queue = "; ".join(f"python experiments/generate.py {COMMON} {job}" for job in jobs)
    log = open(f"results/log_gpu{gpu}.txt", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    processes.append(subprocess.Popen(queue, shell=True, env=env, stdout=log, stderr=subprocess.STDOUT))

Progress (re-run this cell to refresh):

In [ ]:
!tail -n 2 results/log_gpu0.txt results/log_gpu1.txt
!ls -la results

Wait for both queues, then pack the results:

In [ ]:
for p in processes:
    p.wait()
!zip -q -r results.zip results
!ls -la results.zip